# 실험 1. 특성 조합 비교

LightGBM으로 기준 특성(C0), 정류장 ID 추가(C3), 배차간격 제거(D4), 두 변경 조합(R1), 최종 특성(R3)을 비교합니다. 각 구성은 월·화·수·토를 차례로 검증일로 두고 평가합니다.

검증은 학습 데이터 날짜를 하루씩 제외하는 방식으로 진행합니다. 무작위 분할은 사용하지 않습니다.

## 1. 환경 준비

Python 3.12 커널을 선택한 뒤 필요하면 아래 설치 셀을 한 번 실행하고 커널을 재시작합니다.

In [ ]:
import sys
assert (3, 12) <= sys.version_info[:2] <= (3, 14), f"Python 3.12 환경을 선택하세요. 현재: {sys.version.split()[0]}"
%pip install -r ../requirements.txt

## 2. 데이터 불러오기

처음에는 연습용 데이터(practice)로 실행하고, 대회 때는 PHASE를 inal로 변경합니다.

In [ ]:
from pathlib import Path
import pandas as pd

PHASE = 'practice'  # 'practice' 또는 'final'
REPO_ID = {'practice': 'CAMUS-LAB/bis-travel-time-practice',
           'final': 'CAMUS-LAB/bis-travel-time-final'}[PHASE]
DATA = Path('../data') / PHASE
if not (DATA / 'train.parquet').exists():
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=REPO_ID, repo_type='dataset', local_dir=DATA)

train = pd.read_parquet(DATA / 'train.parquet')
test = pd.read_parquet(DATA / 'test.parquet')
train['date'] = train['date'].astype(str)
print('train:', train.shape, '| test:', test.shape)
print('train dates:', sorted(train.date.unique()))
print('test labels available:', 'tt_sec' in test.columns)

## 3. 특성 및 검증 함수

각 실험은 같은 LightGBM 설정을 사용해 특성 변경의 효과만 비교합니다. 날짜별 MAE와 평일·토요일 평균을 확인합니다.

In [ ]:
import numpy as np
import lightgbm as lgb

PEAK = {7, 8, 9, 17, 18, 19}

def add_features(df):
    out = df.copy()
    day = pd.to_datetime(out['date']).dt.dayofweek
    out['is_weekend'] = (day >= 5).astype('int8')
    out['is_peak'] = out['arrive_hour'].isin(PEAK).astype('int8')
    out['interval'] = np.select(
        [day == 5, day == 6], [out['interval_sat'], out['interval_sun']],
        default=out['interval_wd'])
    out['dlat'] = out['to_lat'] - out['from_lat']
    out['dlon'] = out['to_lon'] - out['from_lon']
    return out

BASE_CAT = ['route_id']
BASE_NUM = ['route_type', 'direction', 'interval', 'from_lat', 'from_lon',
            'to_lat', 'to_lon', 'dlat', 'dlon', 'stop_seq', 'dist_m',
            'arrive_hour', 'is_weekend', 'is_peak']
EXPERIMENTS = {
    'C0 기준 특성': (BASE_CAT, BASE_NUM),
    'C3 정류장 ID 추가': (BASE_CAT + ['from_stop', 'to_stop'], BASE_NUM),
    'D4 배차간격 제거': (BASE_CAT, [c for c in BASE_NUM if c != 'interval']),
    'R1 정류장 추가 + 배차간격 제거':
        (BASE_CAT + ['from_stop', 'to_stop'], [c for c in BASE_NUM if c != 'interval']),
    'R3 최종 특성 (is_peak 제거)':
        (BASE_CAT + ['from_stop', 'to_stop'], [c for c in BASE_NUM if c not in {'interval', 'is_peak'}]),
}


def prepare(df, categorical, numeric, categories):
    x = add_features(df)[categorical + numeric].copy()
    for col in categorical:
        x[col] = pd.Categorical(x[col], categories=categories[col])
    return x


def mae(y, pred):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(pred))))


def run_cv(categorical, numeric):
    all_features = add_features(pd.concat([train, test], ignore_index=True))
    categories = {col: pd.Index(all_features[col].dropna().unique()) for col in categorical}
    rows = []
    for val_date in sorted(train.date.unique()):
        fit = train[train.date != val_date]
        val = train[train.date == val_date]
        model = lgb.LGBMRegressor(
            objective='regression_l1', learning_rate=0.05, num_leaves=127,
            min_child_samples=10, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, n_estimators=1500, random_state=0, verbose=-1)
        model.fit(
            prepare(fit, categorical, numeric, categories), fit['tt_sec'],
            eval_set=[(prepare(val, categorical, numeric, categories), val['tt_sec'])],
            eval_metric='l1', callbacks=[lgb.early_stopping(100, verbose=False)])
        pred = model.predict(prepare(val, categorical, numeric, categories))
        rows.append({'date': val_date, 'MAE': mae(val['tt_sec'], pred),
                     'weekday': pd.Timestamp(val_date).dayofweek < 5})
    return pd.DataFrame(rows)

## 4. 실험 실행

R3는 실험 기록의 최종 특성 조합입니다. 이 노트북의 점수는 날짜별 검증 점수이며, 대회 test 점수와 혼동하지 않습니다.

In [ ]:
results = []
details = {}
for name, (categorical, numeric) in EXPERIMENTS.items():
    scores = run_cv(categorical, numeric)
    details[name] = scores
    weekday_mae = scores.loc[scores['weekday'], 'MAE'].mean()
    weekend_mae = scores.loc[~scores['weekday'], 'MAE'].mean()
    results.append({'실험': name, '평일 MAE': weekday_mae,
                    '토요일 MAE': weekend_mae,
                    '4일 평균 MAE': (3 * weekday_mae + weekend_mae) / 4})
comparison = pd.DataFrame(results).sort_values('4일 평균 MAE').reset_index(drop=True)
comparison.round(3)

In [ ]:
for name, scores in details.items():
    print(f'\n{name}')
    print(scores[['date', 'MAE']].to_string(index=False, formatters={'MAE': '{:.3f}'.format}))